In [ ]:
# ── Step 1: Mount Google Drive ──────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── Step 2: Configure paths ──────────────────────────────────────────────────
import pathlib

DRIVE_FOLDER      = pathlib.Path('/content/drive/MyDrive/ShapResearch/Woodelf_package/woodelf_versions_experiment/')
DRIVE_CACHE       = DRIVE_FOLDER / 'method_cache'
EXPERIMENT_NAME   = 'woodelf_versions_experiment'
EXPERIMENT_MODULE = 'benchmarks.woodelf_versions_experiment'

DRIVE_FOLDER.mkdir(parents=True, exist_ok=True)
DRIVE_CACHE.mkdir(parents=True, exist_ok=True)
print(f'Drive folder: {DRIVE_FOLDER}')

In [ ]:
# ── Step 3: Clone treebranchmarks ────────────────────────────────────────────
# woodelf versions are installed automatically by WoodelfVersionApproach
# during the experiment run (pip install --target per ref).
TREEBRANCHMARKS_URL = 'https://github.com/ron-wettenstein/TreeBranchMarks.git'

!git clone {TREEBRANCHMARKS_URL} /content/treebranchmarks

In [ ]:
# ── Step 4: Install treebranchmarks ──────────────────────────────────────────
import sys

# Install the currently-active woodelf (needed for SHAP baseline and model loading)
!pip install -q woodelf
!pip install -q -e /content/treebranchmarks

if '/content/treebranchmarks' not in sys.path:
    sys.path.insert(0, '/content/treebranchmarks')

from treebranchmarks.methods.woodelf_version_method import WoodelfVersionApproach
from treebranchmarks.methods.shap_method import SHAPApproach
print('Imports OK')

In [ ]:
# ── Step 5: Restore method caches from Drive (resume after interruption) ──────
import shutil, pathlib

cache_dir = pathlib.Path(f'/content/treebranchmarks/cache/method_results/{EXPERIMENT_NAME}')
cache_dir.mkdir(parents=True, exist_ok=True)

restored = 0
for f in DRIVE_CACHE.glob('*.json'):
    dest = cache_dir / f.name
    if not dest.exists():
        shutil.copy(f, dest)
        print(f'Restored {f.name} ({f.stat().st_size // 1024} KB)')
        restored += 1
    else:
        print(f'Already present: {f.name}')

if not restored:
    print('No cached results to restore — starting fresh')

In [ ]:
# ── Step 6: Run the experiment ────────────────────────────────────────────────
# WoodelfVersionApproach installs each git ref on first use into
# cache/woodelf_versions/{ref}/ — this happens automatically.
%cd /content/treebranchmarks
!python -u -m {EXPERIMENT_MODULE} --result_location {DRIVE_CACHE}

In [ ]:
# ── Step 7: Save HTML report to Drive and download ───────────────────────────
import shutil, pathlib
from google.colab import files

html = pathlib.Path(f'/content/treebranchmarks/results/{EXPERIMENT_NAME}.html')
dest = DRIVE_FOLDER / html.name
shutil.copy(html, dest)
print(f'Saved to Drive: {dest}')
files.download(str(html))